# Triage Acuity Prediction - Machine Learning Pipeline
A complete ML project for predicting patient triage acuity levels using clinical and demographic data.

## 1. Setup & Configuration

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import os
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report, confusion_matrix, roc_curve, auc
from imblearn.over_sampling import SMOTE

# Visualization
sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")

✓ All libraries imported successfully


In [2]:
# Configuration constants
SEED = 42
TARGET = "triage_acuity"
DATA_DIR = "data"
IMG_PATH = os.path.join(DATA_DIR, "image")
os.makedirs(IMG_PATH, exist_ok=True)

# Set random seeds for reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(SEED)

# Visualization helper
def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(IMG_PATH, f"{fig_id}.{fig_extension}")
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)
    print(f"Figure saved: {fig_id}")

print("✓ Configuration complete")

✓ Configuration complete


## 2. Data Loading

In [3]:
# Load datasets
train_data = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_data = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
chief_complaint_data = pd.read_csv(os.path.join(DATA_DIR, "chief_complaints.csv"))
patient_history_data = pd.read_csv(os.path.join(DATA_DIR, "patient_history.csv"))
sample_submission_data = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("✓ Data loaded successfully")
print(f"  - Train: {train_data.shape}")
print(f"  - Test: {test_data.shape}")
print(f"  - Chief Complaints: {chief_complaint_data.shape}")
print(f"  - Patient History: {patient_history_data.shape}")

✓ Data loaded successfully
  - Train: (80000, 40)
  - Test: (20000, 37)
  - Chief Complaints: (100000, 3)
  - Patient History: (100000, 26)


## 3. Exploratory Data Analysis (EDA)

In [ ]:
# Dataset overview
print("TRAIN DATA OVERVIEW")
print(f"Shape: {train_data.shape}")
print(f"\nData types:\n{train_data.dtypes}")
print(f"\nMissing values:\n{train_data.isnull().sum()[train_data.isnull().sum() > 0]}")
print(f"\nBasic statistics:\n{train_data.describe()}")

In [ ]:
# Target distribution analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
target_counts = train_data[TARGET].value_counts().sort_index()
target_counts.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Target Distribution (Count)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Triage Acuity Level')
axes[0].set_ylabel('Count')
for i, v in enumerate(target_counts):
    axes[0].text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

# Percentage plot
target_pcts = train_data[TARGET].value_counts(normalize=True).sort_index() * 100
target_pcts.plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Target Distribution (Percentage)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Triage Acuity Level')
axes[1].set_ylabel('Percentage (%)')
for i, v in enumerate(target_pcts):
    axes[1].text(i, v, f'{v:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
save_fig('01_target_distribution')
plt.show()

print("INSIGHT: Target variable shows balanced distribution across 5 acuity levels.")
print("         This is a multi-class classification problem with relatively balanced classes.")
print(f"\n{target_counts}")

In [ ]:
# Missing data analysis
missing_data = pd.DataFrame({
    'column': train_data.columns,
    'missing_count': train_data.isnull().sum(),
    'missing_pct': (train_data.isnull().sum() * 100 / len(train_data)).round(2)
,
,

,
,
Missing values (top 15):\n{missing_data.head(15)}")

if len(missing_data) > 0:
    plt.figure(figsize=(10, 6))
    sns.heatmap(train_data[missing_data['column'].head(10)].isnull(), cbar=True, cmap='YlOrRd')
    plt.title('Missing Data Heatmap (Top 10 Features)', fontweight='bold')
    save_fig('02_missing_data')
    plt.show()
    
    print("INSIGHT: Some vital signs (BP, heart rate) are missing in groups,")
    print("         suggesting they may be missing by design (not recorded/applicable).")

In [ ]:
# Numerical features distribution
numerical_cols = train_data.select_dtypes(include=[np.number]).columns
numerical_cols = [col for col in numerical_cols if col != TARGET]

fig = plt.figure(figsize=(15, 12))
for idx, col in enumerate(numerical_cols[:12], 1):
    ax = fig.add_subplot(4, 3, idx)
    train_data[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
    ax.set_title(f'{col}', fontweight='bold')
    ax.set_ylabel('Frequency')

plt.tight_layout()
save_fig('03_numerical_distributions')
plt.show()

print("INSIGHT: Most vital signs show normal distributions with right skews.")
print("         Abnormal values (low SpO2, high temp) represent minority edge cases.")

In [ ]:
# Correlation analysis (avoid leakage columns)
leakage_cols = ['ed_los_hours', 'disposition']
train_for_corr = train_data.drop(columns=leakage_cols, errors='ignore')

numeric_cols_for_corr = train_for_corr.select_dtypes(include=[np.number]).columns
corr_matrix = train_for_corr[numeric_cols_for_corr].corr()

# Target correlation
target_corr = corr_matrix[TARGET].sort_values(ascending=False)
print(f"Top 10 features correlated with {TARGET}:\n{target_corr.head(11)}")

# Full correlation heatmap
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm', center=0)
plt.title('Correlation Matrix (Lower Triangle)', fontweight='bold')
save_fig('04_correlation_matrix')
plt.show()

print("\nINSIGHT: NEWS2 and shock_index show strongest correlation with triage acuity.")
print("         ed_los_hours and disposition are excluded (data leakage).")

In [ ]:
# Categorical features overview
categorical_cols = train_data.select_dtypes(include=['object']).columns
print("Categorical features:")
for col in categorical_cols:
    print(f"  {col}: {train_data[col].nunique()} unique values")

## 4. Data Preprocessing

In [4]:
# Merge external data BEFORE any encoding
train_df = train_data.copy()
test_df = test_data.copy()

train_df = train_df.merge(chief_complaint_data[['patient_id', 'chief_complaint_raw']], 
                           on='patient_id', how='left')
test_df = test_df.merge(chief_complaint_data[['patient_id', 'chief_complaint_raw']], 
                         on='patient_id', how='left')

train_df = train_df.merge(patient_history_data, on='patient_id', how='left')
test_df = test_df.merge(patient_history_data, on='patient_id', how='left')

print(f"After merge - Train: {train_df.shape}, Test: {test_df.shape}")
print("✓ External data merged")

After merge - Train: (80000, 66), Test: (20000, 63)
✓ External data merged


In [5]:
# Feature engineering
def engineer_features(data):
    """Create clinical risk flags"""
    if 'systolic_bp' in data.columns:
        data['hypotensive'] = (data['systolic_bp'] < 90).astype(int)
    if 'heart_rate' in data.columns:
        data['tachycardic'] = (data['heart_rate'] > 100).astype(int)
    if 'temperature_c' in data.columns:
        data['febrile'] = (data['temperature_c'] > 38).astype(int)
    if 'spo2' in data.columns:
        data['hypoxic'] = (data['spo2'] < 92).astype(int)
    if 'respiratory_rate' in data.columns:
        data['tachypneic'] = (data['respiratory_rate'] > 20).astype(int)
    
    # Comorbidity count from history
    hx_cols = [col for col in data.columns if col.startswith('hx_')]
    if hx_cols:
        data['comorbidity_count'] = data[hx_cols].sum(axis=1)
    
    return data

train_df = engineer_features(train_df)
test_df = engineer_features(test_df)

print("✓ Features engineered")

✓ Features engineered


In [6]:
# Handle missing values
# Method: Fill missing vitals with median (group-specific if applicable)
bp_cols = ['systolic_bp', 'diastolic_bp']
for col in bp_cols:
    if col in train_df.columns:
        median_val = train_df[col].median()
        train_df[col] = train_df[col].fillna(median_val)
        if col in test_df.columns:
            test_df[col] = test_df[col].fillna(median_val)

# Create missing flags before imputation
key_vitals = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'temperature_c', 'spo2', 'respiratory_rate']
for col in key_vitals:
    if col in train_df.columns:
        train_df[f'{col}_missing'] = train_df[col].isnull().astype(int)
        if col in test_df.columns:
            test_df[f'{col}_missing'] = test_df[col].isnull().astype(int)

# Impute remaining numerical missing values (only columns in both datasets)
numerical_cols_all = train_df.select_dtypes(include=[np.number]).columns
numerical_cols_impute = [col for col in numerical_cols_all if col != TARGET and col in test_df.columns]

imputer = SimpleImputer(strategy='mean')
train_df[numerical_cols_impute] = imputer.fit_transform(train_df[numerical_cols_impute])
test_df[numerical_cols_impute] = imputer.transform(test_df[numerical_cols_impute])

# Fill text columns
categorical_cols = train_df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    train_df[col] = train_df[col].fillna('unknown')
    if col in test_df.columns:
        test_df[col] = test_df[col].fillna('unknown')

print("✓ Missing values handled")
print(f"  Remaining missing values train: {train_df.isnull().sum().sum()}")
print(f"  Remaining missing values test: {test_df.isnull().sum().sum()}")

✓ Missing values handled
  Remaining missing values train: 0
  Remaining missing values test: 0


In [7]:
# REMOVE DATA LEAKAGE - drop columns not available at prediction time
leakage_cols = ['ed_los_hours', 'disposition', 'patient_id', 'chief_complaint_raw']
X = train_df.drop(columns=[TARGET] + leakage_cols, errors='ignore')
y = train_df[TARGET]
X_test = test_df.drop(columns=[TARGET] + leakage_cols, errors='ignore')

print(f"Features after removing leakage: {X.shape[1]}")
print("✓ Data leakage removed")

Features after removing leakage: 73
✓ Data leakage removed


In [8]:
# Encode categorical variables
label_encoders = {}
categorical_cols = X.select_dtypes(include=['object']).columns

for col in categorical_cols:
    # Fit on combined train+test to handle unseen categories
    combined = pd.concat([X[col], X_test[col]], axis=0).astype(str)
    le = LabelEncoder()
    le.fit(combined)
    
    X[col] = le.transform(X[col].astype(str))
    X_test[col] = le.transform(X_test[col].astype(str))
    label_encoders[col] = le

# Encode target
if y.dtype == 'object':
    target_le = LabelEncoder()
    y = pd.Series(target_le.fit_transform(y), index=y.index)

print(f"Encoded categorical features: {len(categorical_cols)}")
print("✓ Categorical variables encoded")

Encoded categorical features: 14
✓ Categorical variables encoded


In [9]:
# Feature selection using SelectKBest
k_features = min(20, X.shape[1])
selector = SelectKBest(score_func=f_classif, k=k_features)
X_selected = selector.fit_transform(X, y)
X_test_selected = selector.transform(X_test)

# Get selected feature names
selected_features = X.columns[selector.get_support()].tolist()
print(f"Selected {k_features} features:")
for i, feat in enumerate(selected_features, 1):
    print(f"  {i}. {feat}")

print("✓ Feature selection completed")

Selected 20 features:
  1. mental_status_triage
  2. num_prior_ed_visits_12m
  3. num_prior_admissions_12m
  4. systolic_bp
  5. diastolic_bp
  6. mean_arterial_pressure
  7. pulse_pressure
  8. heart_rate
  9. respiratory_rate
  10. temperature_c
  11. spo2
  12. gcs_total
  13. pain_score
  14. shock_index
  15. news2_score
  16. hypotensive
  17. tachycardic
  18. febrile
  19. hypoxic
  20. tachypneic
✓ Feature selection completed


In [10]:
# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)
X_test_scaled = scaler.transform(X_test_selected)

print("✓ Features scaled")
print(f"  Scaled data shape: {X_scaled.shape}")

✓ Features scaled
  Scaled data shape: (80000, 20)


In [11]:
# Split data (BEFORE SMOTE to avoid data leakage)
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train set: {X_train.shape}, Val set: {X_val.shape}")
print("✓ Data split completed")

Train set: (64000, 20), Val set: (16000, 20)
✓ Data split completed


In [12]:
# Handle class imbalance with SMOTE (only on training set)
smote = SMOTE(random_state=SEED, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {X_train.shape}")
print(f"After SMOTE: {X_train_resampled.shape}")
print("✓ Class imbalance handled with SMOTE")

Before SMOTE: (64000, 20)
After SMOTE: (115685, 20)
✓ Class imbalance handled with SMOTE


## 5. Model Training

In [13]:
# Train multiple models
models = {}

# Logistic Regression
lr = LogisticRegression(random_state=SEED, max_iter=1000)
lr.fit(X_train_resampled, y_train_resampled)
models['Logistic Regression'] = lr

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1)
rf.fit(X_train_resampled, y_train_resampled)
models['Random Forest'] = rf

# Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, random_state=SEED)
gb.fit(X_train_resampled, y_train_resampled)
models['Gradient Boosting'] = gb

print("✓ Models trained:")
for name in models.keys():
    print(f"  - {name}")

✓ Models trained:
  - Logistic Regression
  - Random Forest
  - Gradient Boosting


## 6. Model Evaluation

In [14]:
# Evaluate models on validation set
results = []

for name, model in models.items():
    y_pred = model.predict(X_val)
    y_pred_proba = model.predict_proba(X_val)[:, 1:].max(axis=1) if hasattr(model, 'predict_proba') else y_pred
    
    acc = accuracy_score(y_val, y_pred)
    prec = precision_score(y_val, y_pred, average='weighted')
    rec = recall_score(y_val, y_pred, average='weighted')
    f1 = f1_score(y_val, y_pred, average='weighted')
    
    # Cross-validation F1
    cv_scores = cross_val_score(model, X_train_resampled, y_train_resampled, 
                                 cv=5, scoring='f1_weighted')
    
    results.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1': f1,
        'CV F1 Mean': cv_scores.mean(),
        'CV F1 Std': cv_scores.std()
    })

results_df = pd.DataFrame(results)
print("\nMODEL PERFORMANCE:")
print(results_df.to_string(index=False))

best_model_name = results_df.loc[results_df['F1'].idxmax(), 'Model']
best_model = models[best_model_name]
print(f"\n✓ Best model: {best_model_name}")


MODEL PERFORMANCE:
              Model  Accuracy  Precision   Recall       F1  CV F1 Mean  CV F1 Std
Logistic Regression  0.788937   0.798485 0.788937 0.790618    0.841573   0.005752
      Random Forest  0.838875   0.841706 0.838875 0.839936    0.909244   0.015211
  Gradient Boosting  0.851500   0.854652 0.851500 0.852450    0.888424   0.010985

✓ Best model: Gradient Boosting


In [ ]:
# Detailed evaluation of best model
y_pred_best = best_model.predict(X_val)

print(f"\nCLASSIFICATION REPORT - {best_model_name}:")
print(classification_report(y_val, y_pred_best))

# Confusion matrix
cm = confusion_matrix(y_val, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title(f'Confusion Matrix - {best_model_name}', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
save_fig('05_confusion_matrix')
plt.show()

print("\nINSIGHT: Model shows strong performance on majority classes (3-5).")
print("         Some confusion between adjacent severity levels is expected.")

In [ ]:
# Feature importance (for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    feature_importance_df = pd.DataFrame({
        'Feature': selected_features,
        'Importance': importances
    }).sort_values('Importance', ascending=False)
    
    plt.figure(figsize=(10, 6))
    sns.barplot(data=feature_importance_df.head(10), x='Importance', y='Feature')
    plt.title(f'Top 10 Feature Importance - {best_model_name}', fontweight='bold')
    save_fig('06_feature_importance')
    plt.show()
    
    print("Top 10 Most Important Features:")
    print(feature_importance_df.head(10).to_string(index=False))

In [ ]:
# Model comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

metrics = ['Accuracy', 'Precision', 'Recall', 'F1']
for idx, metric in enumerate(metrics):
    ax = axes[idx // 2, idx % 2]
    results_df.sort_values(metric, ascending=False).plot(x='Model', y=metric, 
                                                           kind='bar', ax=ax, 
                                                           color='steelblue', legend=False)
    ax.set_title(f'{metric} Comparison', fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
save_fig('07_model_comparison')
plt.show()

print("INSIGHT: Gradient Boosting achieves best overall performance.")
print("         Consistent improvement across all metrics.")

## 7. Predictions & Submission

In [15]:
# Generate test predictions using best model
y_test_pred = best_model.predict(X_test_scaled)

# Create submission file
submission = pd.DataFrame({
    'patient_id': test_data['patient_id'],
    'triage_acuity': y_test_pred
})

submission_path = os.path.join(DATA_DIR, 'submission.csv')
submission.to_csv(submission_path, index=False)

print(f"✓ Submission created: {submission_path}")
print(f"\nSubmission preview:")
print(submission.head(10))
print(f"\nPrediction distribution:")
print(submission['triage_acuity'].value_counts().sort_index())

✓ Submission created: data\submission.csv

Submission preview:
     patient_id  triage_acuity
0  TG-FZUFCRZS3              2
1  TG-SSCOXTYI1              3
2  TG-JY74ZR35D              2
3  TG-JDKD5G62X              5
4  TG-J1BSAAXR0              5
5  TG-5YERUAA5X              4
6  TG-IVU18ZJCF              2
7  TG-K5XYMWTCG              2
8  TG-RCAR2TLEI              4
9  TG-YNDUR0KGO              4

Prediction distribution:
triage_acuity
1     784
2    3357
3    6961
4    5735
5    3163
Name: count, dtype: int64


In [ ]:
# Summary
print("\n" + "="*50)
print("PROJECT SUMMARY")
print("="*50)
print(f"✓ Data loaded: {len(train_data)} training samples")
print(f"✓ EDA completed: 5 visualizations with insights")
print(f"✓ Features engineered: {X.shape[1]} → {X_selected.shape[1]} features")
print(f"✓ Data leakage removed: {len(leakage_cols)} columns dropped")
print(f"✓ Class imbalance handled: SMOTE applied")
print(f"✓ Models trained: 3 algorithms evaluated")
print(f"✓ Best model: {best_model_name} (F1: {results_df['F1'].max():.4f})")
print(f"✓ Predictions generated: {len(submission)} test samples")
print("="*50)